In [4]:
import sys
from pathlib import Path

# Notebook liegt in /notebooks -> Parent ist Projekt-Root
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import config
config.ensure_dirs()

print("CONFIG GELADEN")
print("Project root:", config.PROJECT_ROOT)
print("DATA_RAW:", config.DATA_RAW)
print("DATA_PROCESSED:", config.DATA_PROCESSED)


CONFIG GELADEN
Project root: C:\Users\Noah\PG3_Royalty_Python_Model
DATA_RAW: C:\Users\Noah\PG3_Royalty_Python_Model\Data\raw
DATA_PROCESSED: C:\Users\Noah\PG3_Royalty_Python_Model\Data\processed


In [5]:
import pandas as pd

excel_files = sorted(config.DATA_RAW.glob("*.xlsx"))
print("Gefundene Excel-Files:", [f.name for f in excel_files])

excel_path = excel_files[0]  # wenn nur 1 drin ist, passt das
print("Excel wird geladen:", excel_path)

df = pd.read_excel(excel_path, sheet_name="07_Export")

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()


Gefundene Excel-Files: ['PG3_Royalty_Model_GroundTruth_16-02-2026_v1.xlsx.xlsx']
Excel wird geladen: C:\Users\Noah\PG3_Royalty_Python_Model\Data\raw\PG3_Royalty_Model_GroundTruth_16-02-2026_v1.xlsx.xlsx
Rows: 18
Columns: ['Scenario', 'FY', 'Net_CF_to_Consortium', 'Op_Fee', 'Consortium_Fees', 'CF_after_Fees', 'Interest_Rate', 'Interest_Cost', 'Mandatory_Amort', 'Recap_DeltaDebt', 'Cash_Sweep', 'Debt_End', 'FCF_for_Distribution', 'NAV_Multiple', 'NAV', 'LTV', 'PG_Share', 'Equity_Ticket', 'Equity_CF']


,Scenario,FY,Net_CF_to_Consortium,Op_Fee,Consortium_Fees,CF_after_Fees,Interest_Rate,Interest_Cost,Mandatory_Amort,Recap_DeltaDebt,Cash_Sweep,Debt_End,FCF_for_Distribution,NAV_Multiple,NAV,LTV,PG_Share,Equity_Ticket,Equity_CF
0,Base,2024,46.8,2.340,0.2,44.260,0.062,-20.15000,0.0,0.00,0.0,325.00,24.11000,14.0,655.20,0.496032,0.295394,99.4,-92.278056
1,Base,2025,50.3,2.515,0.0,47.785,0.056,-18.20000,0.0,0.00,0.0,325.00,29.58500,14.0,704.20,0.461517,0.295394,99.4,8.739224
2,Base,2026,53.7,2.685,0.0,51.015,0.057,-18.52500,0.0,126.08,0.0,451.08,32.49000,14.0,751.80,0.600000,0.295394,99.4,9.597343
3,Base,2027,56.3,2.815,0.0,53.485,0.057,-25.71156,0.0,0.00,0.0,451.08,27.77344,14.0,788.20,0.572291,0.295394,99.4,8.204101
4,Base,2028,58.3,2.915,0.0,55.385,0.059,-26.61372,0.0,0.00,0.0,451.08,28.77128,13.6,792.88,0.568913,0.295394,99.4,8.498857


In [6]:
valid_scenarios = {"Base", "Flat", "Downside"}

df_clean = df[df["Scenario"].isin(valid_scenarios)].copy()

df_clean["FY"] = pd.to_numeric(df_clean["FY"], errors="coerce")
df_clean = df_clean.dropna(subset=["FY"])
df_clean["FY"] = df_clean["FY"].astype(int)

print("Rows (clean):", len(df_clean))
print("Scenarios (clean):", df_clean["Scenario"].unique())
print("FY min:", df_clean["FY"].min(), "| FY max:", df_clean["FY"].max(), "| Unique FY:", df_clean["FY"].nunique())

print("\nMissing values per column (clean):")
print(df_clean.isna().sum())


Rows (clean): 16
Scenarios (clean): <StringArray>
['Base']
Length: 1, dtype: str
FY min: 2024 | FY max: 2039 | Unique FY: 16

Missing values per column (clean):
Scenario                0
FY                      0
Net_CF_to_Consortium    0
Op_Fee                  0
Consortium_Fees         0
CF_after_Fees           0
Interest_Rate           0
Interest_Cost           0
Mandatory_Amort         0
Recap_DeltaDebt         0
Cash_Sweep              0
Debt_End                0
FCF_for_Distribution    0
NAV_Multiple            0
NAV                     0
LTV                     0
PG_Share                0
Equity_Ticket           0
Equity_CF               0
dtype: int64


In [7]:
df_clean.groupby("Scenario")["FY"].agg(["min", "max", "nunique"])


,min,max,nunique
Scenario,,,
Base,2024,2039,16


In [8]:
df_clean.to_csv(config.GROUND_TRUTH_CSV, index=False)
print("Saved to:", config.GROUND_TRUTH_CSV)


Saved to: C:\Users\Noah\PG3_Royalty_Python_Model\Data\processed\ground_truth_clean.csv
